<a href="https://colab.research.google.com/github/CMPE-255-G5/times-series-anomaly-detection-on-twitter-data/blob/main/notebooks/Twitter_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Load data

In [82]:
# packages
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [63]:
# Download latest version
path = kagglehub.dataset_download("julienjta/twitter-mentions-volumes")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'twitter-mentions-volumes' dataset.
Path to dataset files: /kaggle/input/twitter-mentions-volumes


In [64]:
# List files in the downloaded directory
print(os.listdir(path))

['dataset.csv']


In [65]:
# Construct the full path to the dataset.csv file
file_path = os.path.join(path, 'dataset.csv')

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
display(df.head())

,timestamp,Apple,Amazon,Salesforce,CVS,Facebook,Google,IBM,Coca-Cola,Pfizer,UPS
0,2015-02-26 21:42:53,104,57.0,11,0.0,53.0,35.0,7.0,8.0,3.0,2.0
1,2015-02-26 21:47:53,100,43.0,10,0.0,64.0,41.0,4.0,8.0,2.0,2.0
2,2015-02-26 21:52:53,99,55.0,3,0.0,49.0,32.0,14.0,5.0,2.0,4.0
3,2015-02-26 21:57:53,154,64.0,4,0.0,48.0,36.0,6.0,13.0,36.0,3.0
4,2015-02-26 22:02:53,120,93.0,9,0.0,22.0,32.0,1.0,22.0,8.0,5.0


In [66]:
df.shape

(15902, 11)

In [67]:
# Convert timestamp to date time format
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [68]:
# Check min date, max date
min(df['timestamp']), max(df['timestamp'])

(Timestamp('2015-02-26 21:42:53'), Timestamp('2015-04-23 02:47:53'))

In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15902 entries, 0 to 15901
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   timestamp   15902 non-null  datetime64[ns]
 1   Apple       15902 non-null  int64         
 2   Amazon      15831 non-null  float64       
 3   Salesforce  15902 non-null  int64         
 4   CVS         15853 non-null  float64       
 5   Facebook    15833 non-null  float64       
 6   Google      15842 non-null  float64       
 7   IBM         15893 non-null  float64       
 8   Coca-Cola   15851 non-null  float64       
 9   Pfizer      15858 non-null  float64       
 10  UPS         15866 non-null  float64       
dtypes: datetime64[ns](1), float64(8), int64(2)
memory usage: 1.3 MB


# 2. Data Preprocessing

## 2.1. Duplicates

In [70]:
# Check duplicates
df.duplicated().sum()

np.int64(0)

## 2.2 Train - Val - Test Split

In [71]:
# Sort chronologically
df = df.sort_values('timestamp').reset_index(drop=True)

In [72]:
# Split train:validation:test with the ratio 70:15:15
n = len(df)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

print(n, train_end, val_end)

15902 11131 13516


In [73]:
# Split the data
train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]

In [74]:
# Check shapes of train, validation and test
train.shape, val.shape, test.shape

((11131, 11), (2385, 11), (2386, 11))

In [75]:
# Check min, max for train, val, test
for name, df in [('train', train), ('val', val), ('test', test)]:
    print(f'{name}: {min(df['timestamp'])} → {max(df['timestamp'])}')

train: 2015-02-26 21:42:53 → 2015-04-06 13:12:53
val: 2015-04-06 13:17:53 → 2015-04-14 19:57:53
test: 2015-04-14 20:02:53 → 2015-04-23 02:47:53


## 2.3. Missing Values

In [76]:
# Function to check missing values
def summarize_missing(train, val, test):
    missing_summary = pd.DataFrame(
        {
            "train": train.isnull().sum(),
            "val": val.isnull().sum(),
            "test": test.isnull().sum(),
        }
    )
    return missing_summary

# Check missing values
summarize_missing(train, val, test)

,train,val,test
timestamp,0,0,0
Apple,0,0,0
Amazon,0,0,71
Salesforce,0,0,0
CVS,0,0,49
Facebook,0,0,69
Google,0,0,60
IBM,0,0,9
Coca-Cola,0,0,51
Pfizer,0,0,44


In [77]:
# Check missing values in test set
test[test['Amazon'].isnull()]

,timestamp,Apple,Amazon,Salesforce,CVS,Facebook,Google,IBM,Coca-Cola,Pfizer,UPS
15831,2015-04-22 20:57:53,120,NaN,9,1.0,78.0,50.0,3.0,16.0,1.0,1.0
15832,2015-04-22 21:02:53,109,NaN,7,0.0,117.0,61.0,4.0,21.0,2.0,1.0
15833,2015-04-22 21:07:53,44,NaN,6,0.0,NaN,56.0,2.0,15.0,0.0,0.0
15834,2015-04-22 21:12:53,74,NaN,4,0.0,NaN,59.0,1.0,13.0,0.0,2.0
15835,2015-04-22 21:17:53,60,NaN,5,0.0,NaN,57.0,6.0,13.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
15897,2015-04-23 02:27:53,44,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15898,2015-04-23 02:32:53,45,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15899,2015-04-23 02:37:53,48,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15900,2015-04-23 02:42:53,26,NaN,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Missing values were found in the test set, concentrated in the last two days of the dataset and affecting only specific brand columns (Amazon, CVS, Facebook,
Google, IBM, Coca-Cola, Pfizer, UPS). This suggests brand-specific data collection issues rather than a full pipeline failure. Values were imputed using forward
fill (ffill) based on train set statistics to avoid data leakage.

In [94]:
# Copy train and val for consistent naming
train_imp = train.copy()
val_imp = val.copy()

# Calculate median of train data. Use median instead of mean to ignore the spikes
train_median = train.median()

# Impute the missing value
# ffill: forward fill carries the last known value forward
# if the very first row is missing then use train_median
test_imp = test.fillna(method='ffill').fillna(train_median)

/tmp/ipykernel_5942/2230151619.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test_imp = test.fillna(method='ffill').fillna(train_median)


In [79]:
test_imp.tail(71)

,timestamp,Apple,Amazon,Salesforce,CVS,Facebook,Google,IBM,Coca-Cola,Pfizer,UPS
15831,2015-04-22 20:57:53,120,50.0,9,1.0,78.0,50.0,3.0,16.0,1.0,1.0
15832,2015-04-22 21:02:53,109,50.0,7,0.0,117.0,61.0,4.0,21.0,2.0,1.0
15833,2015-04-22 21:07:53,44,50.0,6,0.0,117.0,56.0,2.0,15.0,0.0,0.0
15834,2015-04-22 21:12:53,74,50.0,4,0.0,117.0,59.0,1.0,13.0,0.0,2.0
15835,2015-04-22 21:17:53,60,50.0,5,0.0,117.0,57.0,6.0,13.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
15897,2015-04-23 02:27:53,44,50.0,1,0.0,117.0,72.0,1.0,20.0,0.0,3.0
15898,2015-04-23 02:32:53,45,50.0,4,0.0,117.0,72.0,1.0,20.0,0.0,3.0
15899,2015-04-23 02:37:53,48,50.0,3,0.0,117.0,72.0,1.0,20.0,0.0,3.0
15900,2015-04-23 02:42:53,26,50.0,8,0.0,117.0,72.0,1.0,20.0,0.0,3.0


In [80]:
# Check missing values again
summarize_missing(train_imp, val_imp, test_imp)

,train,val,test
timestamp,0,0,0
Apple,0,0,0
Amazon,0,0,0
Salesforce,0,0,0
CVS,0,0,0
Facebook,0,0,0
Google,0,0,0
IBM,0,0,0
Coca-Cola,0,0,0
Pfizer,0,0,0


## 2.4 Feature Scaling

Scaling for distance-based models (e.g. one-class SVM) or deep learning models (autoencoders).

In [81]:
# Check basic desciptive statistics
train_imp.describe().T

,count,mean,min,25%,50%,75%,max,std
timestamp,11131,2015-03-18 05:27:52.999999744,2015-02-26 21:42:53,2015-03-08 13:35:23,2015-03-18 05:27:53,2015-03-27 21:20:23,2015-04-06 13:12:53,NaN
Apple,11131.0,80.82679,0.0,28.0,44.0,73.0,13479.0,282.459239
Amazon,11131.0,54.206989,0.0,37.0,51.0,66.0,1673.0,32.331066
Salesforce,11131.0,3.465637,0.0,1.0,2.0,5.0,209.0,5.064131
CVS,11131.0,0.343455,0.0,0.0,0.0,0.0,50.0,1.026337
Facebook,11131.0,17.87692,0.0,9.0,14.0,22.0,1258.0,21.169324
Google,11131.0,20.874225,0.0,11.0,16.0,25.0,465.0,19.911786
IBM,11131.0,4.098104,0.0,1.0,3.0,6.0,84.0,4.419764
Coca-Cola,11131.0,11.071332,0.0,4.0,8.0,13.0,531.0,17.018882
Pfizer,11131.0,0.846285,0.0,0.0,0.0,1.0,36.0,1.452822


In [92]:
# Standard Normalization Scaling
scaler = StandardScaler()

# Select only brand columns for scaling
scale_cols = [c for c in train_imp.columns if c != 'timestamp']

# Function to scale
def scale_split(df, scaler, cols, fit=False):
    array = scaler.fit_transform(df[cols]) if fit else scaler.transform(df[cols])
    return pd.DataFrame(array, columns=cols, index=df.index).reset_index()

# Scale
train_scaled = scale_split(train_imp, scaler, scale_cols, fit=True)
val_scaled = scale_split(val_imp, scaler, scale_cols)
test_scaled = scale_split(test_imp, scaler, scale_cols)

In [93]:
# Sanity check
train_scaled.describe().T

,count,mean,min,25%,50%,75%,max,std
timestamp,11131,2015-03-18 05:27:52.999999744,2015-02-26 21:42:53,2015-03-08 13:35:23,2015-03-18 05:27:53,2015-03-27 21:20:23,2015-04-06 13:12:53,NaN
Apple,11131.0,0.0,-0.286167,-0.187033,-0.130385,-0.027711,47.436137,1.000045
Amazon,11131.0,0.0,-1.676698,-0.532236,-0.099197,0.364774,50.071523,1.000045
Salesforce,11131.0,0.0,-0.68438,-0.486904,-0.289428,0.303,40.588124,1.000045
CVS,11131.0,-0.0,-0.334657,-0.334657,-0.334657,-0.334657,48.384496,1.000045
Facebook,11131.0,0.0,-0.844511,-0.419348,-0.183147,0.194775,58.583769,1.000045
Google,11131.0,0.0,-1.048382,-0.495921,-0.244802,0.207212,22.305669,1.000045
IBM,11131.0,-0.0,-0.927264,-0.700998,-0.248464,0.430335,18.079131,1.000045
Coca-Cola,11131.0,0.0,-0.650562,-0.415518,-0.180474,0.11333,30.551479,1.000045
Pfizer,11131.0,-0.0,-0.582537,-0.582537,-0.582537,0.105809,24.197939,1.000045


# 3. EDA

# 4. Feature Engineering